In [1]:
import gc
import os
import sys
import math

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
%load_ext autoreload
%autoreload 2
from drGAT import drGAT

from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /gpfs/gsfs12/users/inouey2/conda/envs/genex/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:72: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /gpfs/gsfs12/users/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at23SavedTensorDefaultHooks11set_tracingEb
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:99: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. S

In [3]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg, _, _, _ = load_data('nci')

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|██████████| 25/25 [00:02<00:00, 10.11it/s]


Done!


In [4]:
PATH = "../nci_data/"
method = "GATv2"
study = optuna.create_study(
    storage=f"sqlite:///{method}_small.sqlite3",
    study_name=method,
    load_if_exists=True,
)

[I 2025-04-16 12:26:39,974] Using an existing study with name 'GATv2' instead of creating a new one.


In [5]:
def auto_convert_params(params, nan_replace=None):
    """Convert parameter types automatically

    Args:
        params (dict): Parameter dictionary before conversion
        nan_replace: Replacement value for NaN (default None)

    Returns:
        dict: Parameter dictionary after type conversion
    """
    converted = {}
    for k, v in params.items():
        if isinstance(v, float) and math.isnan(v):
            converted[k] = nan_replace
        elif isinstance(v, float) and v.is_integer():
            converted[k] = int(v)
        else:
            converted[k] = v
    return converted

In [6]:
params = study.best_trials[0].params
params = auto_convert_params(params, nan_replace=0)
params.update(
    {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "epochs": 5,
        "gnn_layer": method,
    }
)
params

{'dropout1': 0.2,
 'dropout2': 0.1,
 'hidden1': 305,
 'hidden2': 85,
 'hidden3': 79,
 'heads': 4,
 'activation': 'gelu',
 'optimizer': 'Adam',
 'lr': 0.00047762282267892084,
 'weight_decay': 6.656420345368285e-05,
 'scheduler': None,
 'n_drug': 1005,
 'n_cell': 60,
 'n_gene': 2582,
 'epochs': 5,
 'gnn_layer': 'GATv2'}

In [7]:
k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

for seed, (train_index, test_index) in enumerate(kfold.split(np.arange(pos_num))):
    sampler = RandomSampler(
        drugAct.T,
        train_index,
        test_index,
        null_mask.T,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed
    )

    break

In [8]:
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, average_precision_score,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from torch.amp import GradScaler, autocast
from torch.nn import Dropout, Linear, Module
from torch.optim import lr_scheduler
from torch_geometric.nn import GATConv, GATv2Conv, GraphNorm, TransformerConv
from tqdm import tqdm

In [9]:
hidden1: int = int(params["hidden1"])
hidden2: int = int(params["hidden2"])
hidden3: int = int(params["hidden3"])
heads: int = int(params["heads"])

linear_drug = Linear(int(params["n_drug"]), hidden1).to(device)
linear_cell = Linear(int(params["n_cell"]), hidden1).to(device)
linear_gene = Linear(int(params["n_gene"]), hidden1).to(device)

n_layers: int = int(params.get("n_layers", 2))

In [10]:
in_channels = [hidden1] + [hidden2 * heads] * (n_layers - 1)
out_channels = [hidden2] * (n_layers - 1) + [hidden3]

In [11]:
layer1 = TransformerConv(
    in_channels[0], out_channels[0], heads=heads, edge_dim=1
).to(device)

In [12]:
tensor = drGAT.get_data_dict(sampler, device)

In [13]:
drug = linear_drug(tensor['drug'])
cell = linear_cell(tensor['cell'])
gene = linear_gene(tensor['gene'])

In [14]:
x = torch.concat(
    [drug, cell, gene]
)
x

tensor([[-0.0591, -0.1671, -0.3005,  ...,  0.4835, -0.2671, -0.0136],
        [-0.0522, -0.1533, -0.3463,  ...,  0.5688, -0.2822, -0.0390],
        [-0.0593, -0.1611, -0.3545,  ...,  0.6048, -0.2976, -0.0405],
        ...,
        [-0.1304,  0.0483,  0.1199,  ..., -0.0455,  0.0430, -0.0276],
        [-0.1059,  0.0687,  0.1022,  ..., -0.0586,  0.0641,  0.0120],
        [-0.2181,  0.1049,  0.1027,  ..., -0.1041,  0.1340, -0.0273]],
       device='cuda:0', grad_fn=<CatBackward0>)

In [15]:
x.shape

torch.Size([3647, 305])

In [16]:
edge_attr = tensor['edge_attr']
edge_attr = edge_attr.unsqueeze(-1)
edge_attr = edge_attr.to(torch.float32).to(device)
edge_index = tensor['edge_index'].to(device)

In [17]:
torch.cuda.empty_cache()

In [18]:
_, attention = layer1(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr,
    return_attention_weights=True,
)
torch.cuda.empty_cache()

In [47]:
tmp = attention[1].cpu().detach().numpy()
edge = attention[0].cpu().detach().numpy()

In [48]:
tmp.shape

(5384794, 4)

In [49]:
tmp.mean(axis=1)

array([0.00058134, 0.00067892, 0.00050392, ..., 0.00063045, 0.00050239,
       0.00050447], dtype=float32)

In [54]:
idx = max(edge[0]) + 1
graph = np.zeros((idx, idx))
graph

In [59]:
graph = np.zeros((idx, idx))
graph

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [60]:
graph[edge[0], edge[1]] = tmp.mean(axis=1)

In [63]:
tmp = pd.DataFrame(graph)
tmp

,0,1,2,3,4,5,6,7,8,9,...,3637,3638,3639,3640,3641,3642,3643,3644,3645,3646
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000958,0.000964,0.000983,0.000959,0.000976,0.000966,0.000966,0.000973,0.000975,0.000948
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000960,0.000966,0.000985,0.000961,0.000978,0.000968,0.000969,0.000975,0.000978,0.000950
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000960,0.000967,0.000985,0.000962,0.000979,0.000968,0.000970,0.000976,0.000978,0.000950
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000959,0.000965,0.000984,0.000960,0.000977,0.000967,0.000968,0.000975,0.000977,0.000949
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000960,0.000967,0.000986,0.000962,0.000979,0.000968,0.000970,0.000976,0.000978,0.000950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3642,0.000382,0.000383,0.000383,0.000382,0.000384,0.000383,0.000385,0.000383,0.000384,0.000385,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3643,0.000381,0.000381,0.000381,0.000380,0.000382,0.000381,0.000383,0.000382,0.000382,0.000383,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3644,0.000382,0.000382,0.000383,0.000382,0.000384,0.000383,0.000384,0.000383,0.000383,0.000384,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3645,0.000380,0.000381,0.000381,0.000380,0.000382,0.000381,0.000383,0.000382,0.000382,0.000383,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [67]:
p = tmp.loc[:1005, 1005+60:]
p

,1065,1066,1067,1068,1069,1070,1071,1072,1073,1074,...,3637,3638,3639,3640,3641,3642,3643,3644,3645,3646
0,0.000987,0.000974,0.000989,0.000986,0.000963,0.000970,0.000974,0.000974,0.000972,0.000962,...,0.000958,0.000964,0.000983,0.000959,0.000976,0.000966,0.000966,0.000973,0.000975,0.000948
1,0.000989,0.000976,0.000991,0.000989,0.000965,0.000971,0.000977,0.000976,0.000975,0.000963,...,0.000960,0.000966,0.000985,0.000961,0.000978,0.000968,0.000969,0.000975,0.000978,0.000950
2,0.000989,0.000977,0.000991,0.000990,0.000965,0.000972,0.000978,0.000977,0.000975,0.000964,...,0.000960,0.000967,0.000985,0.000962,0.000979,0.000968,0.000970,0.000976,0.000978,0.000950
3,0.000988,0.000976,0.000990,0.000988,0.000964,0.000971,0.000976,0.000975,0.000974,0.000963,...,0.000959,0.000965,0.000984,0.000960,0.000977,0.000967,0.000968,0.000975,0.000977,0.000949
4,0.000989,0.000977,0.000992,0.000990,0.000966,0.000972,0.000978,0.000977,0.000976,0.000964,...,0.000960,0.000967,0.000986,0.000962,0.000979,0.000968,0.000970,0.000976,0.000978,0.000950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1001,0.000987,0.000974,0.000989,0.000987,0.000963,0.000970,0.000975,0.000975,0.000973,0.000962,...,0.000958,0.000964,0.000983,0.000959,0.000977,0.000966,0.000966,0.000973,0.000976,0.000948
1002,0.000987,0.000974,0.000988,0.000987,0.000963,0.000970,0.000974,0.000974,0.000972,0.000961,...,0.000958,0.000964,0.000983,0.000959,0.000976,0.000966,0.000966,0.000974,0.000975,0.000948
1003,0.000986,0.000974,0.000988,0.000986,0.000963,0.000970,0.000974,0.000974,0.000972,0.000961,...,0.000957,0.000964,0.000982,0.000958,0.000976,0.000966,0.000966,0.000973,0.000975,0.000947
1004,0.000987,0.000975,0.000989,0.000987,0.000964,0.000970,0.000975,0.000975,0.000973,0.000963,...,0.000959,0.000965,0.000984,0.000959,0.000977,0.000967,0.000966,0.000974,0.000976,0.000949
